# Linear and Binary Search Algorithms

This notebook covers fundamental search algorithms, from the simple linear search to various binary search variants.

## Table of Contents
1. [Linear Search](#1-linear-search)
2. [Binary Search (Classic)](#2-binary-search-classic)
3. [Binary Search Variants](#3-binary-search-variants)
   - Find First Occurrence
   - Find Last Occurrence
   - Lower Bound / Upper Bound
   - Search in Rotated Sorted Array
4. [Performance Comparison](#4-performance-comparison)
5. [Common Pitfalls](#5-common-pitfalls)

---[< Previous: Linear Time Sorting Algorithms](../02_Sorting_Algorithms/02_linear_sorts.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Linked Lists >](../04_Linear_Data_Structures/01_linked_lists.ipynb)---

In [ ]:
# Required imports for all sections
import numpy as np
import time
import random
import matplotlib.pyplot as plt
from typing import List, Optional, Tuple

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

---
## 1. Linear Search

### Theory

**When to use?**
- When the array is **unsorted** or cannot be sorted
- When the array is **small** (< 10-20 elements)
- When you need to search **only once** (sorting + binary search is overkill)
- When elements are stored in a **linked list** (no random access)

**Prerequisites:**
- None! Works on any collection

**Key insight:**
- Check each element one by one until found or end of array
- Simple but inefficient for large datasets

### ASCII Art Visualization

```
Find 7 in [3, 1, 4, 1, 5, 9, 2, 6, 7]

Step 1: [3, 1, 4, 1, 5, 9, 2, 6, 7]
         ↑
         3 ≠ 7 → continue

Step 2: [3, 1, 4, 1, 5, 9, 2, 6, 7]
            ↑
            1 ≠ 7 → continue

Step 3: [3, 1, 4, 1, 5, 9, 2, 6, 7]
               ↑
               4 ≠ 7 → continue

...continues checking each element...

Step 9: [3, 1, 4, 1, 5, 9, 2, 6, 7]
                                 ↑
                                 7 = 7 → FOUND at index 8!

Summary:
         [3, 1, 4, 1, 5, 9, 2, 6, 7]
          ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ✓
          1  2  3  4  5  6  7  8  9 comparisons
```

### Complexity Table

| Metric | Complexity | Notes |
|--------|------------|-------|
| Time (Best) | O(1) | Element at first position |
| Time (Average) | O(n/2) = O(n) | Element in the middle |
| Time (Worst) | O(n) | Element at end or not found |
| Space | O(1) | Only a few variables needed |

### Implementation

In [ ]:
def linear_search(arr: List[int], target: int) -> int:
    """
    Search for target in array using linear search.
    
    Args:
        arr: List of elements to search through
        target: Element to find
    
    Returns:
        Index of target if found, -1 otherwise
    
    Time Complexity: O(n)
    Space Complexity: O(1)
    """
    for i in range(len(arr)):
        if arr[i] == target:
            return i
    return -1


def linear_search_with_sentinel(arr: List[int], target: int) -> int:
    """
    Linear search optimized with sentinel value.
    Eliminates bound checking in the loop.
    
    Args:
        arr: List of elements (will be temporarily modified)
        target: Element to find
    
    Returns:
        Index of target if found, -1 otherwise
    """
    if not arr:
        return -1
    
    # Save the last element and place sentinel
    last = arr[-1]
    arr[-1] = target
    
    i = 0
    while arr[i] != target:
        i += 1
    
    # Restore the last element
    arr[-1] = last
    
    # Check if we found the target or just the sentinel
    if i < len(arr) - 1 or arr[-1] == target:
        return i
    return -1

### Examples

In [ ]:
def linear_search_verbose(arr: List[int], target: int) -> int:
    """Linear search with step-by-step output."""
    print(f"Searching for {target} in {arr}")
    print("-" * 50)
    
    for i in range(len(arr)):
        print(f"Step {i+1}: Checking index {i}, value = {arr[i]}", end="")
        if arr[i] == target:
            print(f" ✓ FOUND!")
            return i
        print(f" ✗ Continue")
    
    print(f"Element {target} not found after {len(arr)} comparisons")
    return -1

# Test cases
print("=" * 60)
print("TEST 1: Normal case")
print("=" * 60)
result = linear_search_verbose([3, 1, 4, 1, 5, 9, 2, 6, 7], 7)
print(f"\nResult: index {result}\n")

In [ ]:
print("=" * 60)
print("TEST 2: Element at beginning (best case)")
print("=" * 60)
result = linear_search_verbose([7, 1, 4, 1, 5, 9, 2, 6, 3], 7)
print(f"\nResult: index {result}\n")

In [ ]:
print("=" * 60)
print("TEST 3: Edge cases")
print("=" * 60)

# Empty array
print("\nEmpty array:")
print(f"linear_search([], 5) = {linear_search([], 5)}")

# Single element - found
print("\nSingle element (found):")
print(f"linear_search([5], 5) = {linear_search([5], 5)}")

# Single element - not found
print("\nSingle element (not found):")
print(f"linear_search([3], 5) = {linear_search([3], 5)}")

# Element not in array
print("\nElement not found:")
print(f"linear_search([1, 2, 3, 4], 10) = {linear_search([1, 2, 3, 4], 10)}")

---
## 2. Binary Search (Classic)

### Theory

**When to use?**
- When the array is **sorted**
- When you need to search **multiple times** (sorting cost is amortized)
- When the array is **large** and O(n) is too slow
- When you need to find insertion points or bounds

**Prerequisites:**
- Array must be **sorted** in ascending (or descending) order
- Random access to elements (arrays, not linked lists)

**Key insight:**
- Divide search space in half with each comparison
- If middle element < target → search right half
- If middle element > target → search left half
- Each step eliminates half of remaining elements

### ASCII Art Visualization

```
Find 7 in [1, 2, 3, 4, 5, 6, 7, 8, 9]
          indices: 0  1  2  3  4  5  6  7  8

Step 1: [1, 2, 3, 4, 5, 6, 7, 8, 9]
         L           M           R
         ↑           ↑           ↑
       left=0     mid=4      right=8
       
       arr[4] = 5
       5 < 7 → target is in RIGHT half
       Update: left = mid + 1 = 5

Step 2: [1, 2, 3, 4, 5, 6, 7, 8, 9]
                     L  M     R
                     ↑  ↑     ↑
                  left=5  mid=6  right=8
       
       arr[6] = 7
       7 = 7 → FOUND at index 6!

Summary: Only 2 comparisons vs 7 for linear search!

Search space reduction:
Step 1: 9 elements → 4 elements (eliminated 5)
Step 2: 4 elements → FOUND
```

### Complexity Table

| Metric | Complexity | Notes |
|--------|------------|-------|
| Time (Best) | O(1) | Target is at middle |
| Time (Average) | O(log n) | Halving search space |
| Time (Worst) | O(log n) | Target at extreme or not found |
| Space (Iterative) | O(1) | Only a few variables |
| Space (Recursive) | O(log n) | Call stack depth |

### Implementation

In [ ]:
def binary_search_iterative(arr: List[int], target: int) -> int:
    """
    Search for target in sorted array using iterative binary search.
    
    Args:
        arr: Sorted list of elements
        target: Element to find
    
    Returns:
        Index of target if found, -1 otherwise
    
    Time Complexity: O(log n)
    Space Complexity: O(1)
    """
    left, right = 0, len(arr) - 1
    
    while left <= right:
        # Prevent integer overflow: mid = left + (right - left) // 2
        mid = (left + right) // 2
        
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1  # Search right half
        else:
            right = mid - 1  # Search left half
    
    return -1  # Not found


def binary_search_recursive(arr: List[int], target: int, 
                            left: int = None, right: int = None) -> int:
    """
    Search for target in sorted array using recursive binary search.
    
    Args:
        arr: Sorted list of elements
        target: Element to find
        left: Left boundary (inclusive)
        right: Right boundary (inclusive)
    
    Returns:
        Index of target if found, -1 otherwise
    
    Time Complexity: O(log n)
    Space Complexity: O(log n) - call stack
    """
    # Initialize boundaries on first call
    if left is None:
        left = 0
    if right is None:
        right = len(arr) - 1
    
    # Base case: search space exhausted
    if left > right:
        return -1
    
    mid = (left + right) // 2
    
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, right)
    else:
        return binary_search_recursive(arr, target, left, mid - 1)

### Examples

In [ ]:
def binary_search_verbose(arr: List[int], target: int) -> int:
    """Binary search with step-by-step visualization."""
    print(f"Searching for {target} in {arr}")
    print("-" * 60)
    
    left, right = 0, len(arr) - 1
    step = 1
    
    while left <= right:
        mid = (left + right) // 2
        
        # Visualization
        print(f"\nStep {step}:")
        print(f"  Array:  {arr}")
        print(f"  Indices: ", end="")
        for i in range(len(arr)):
            print(f"{i:>3}", end="")
        print()
        
        # Show pointers
        pointer_line = "           "
        for i in range(len(arr)):
            if i == left and i == mid and i == right:
                pointer_line += "LMR"
            elif i == left and i == mid:
                pointer_line += "LM "
            elif i == mid and i == right:
                pointer_line += "MR "
            elif i == left:
                pointer_line += "L  "
            elif i == mid:
                pointer_line += "M  "
            elif i == right:
                pointer_line += "R  "
            else:
                pointer_line += "   "
        print(pointer_line)
        
        print(f"  left={left}, mid={mid}, right={right}")
        print(f"  arr[{mid}] = {arr[mid]}")
        
        if arr[mid] == target:
            print(f"  {arr[mid]} == {target} → FOUND at index {mid}!")
            return mid
        elif arr[mid] < target:
            print(f"  {arr[mid]} < {target} → search RIGHT half")
            left = mid + 1
        else:
            print(f"  {arr[mid]} > {target} → search LEFT half")
            right = mid - 1
        
        step += 1
    
    print(f"\nElement {target} not found after {step-1} steps")
    return -1

# Test
print("=" * 70)
print("TEST: Finding 7 in sorted array")
print("=" * 70)
result = binary_search_verbose([1, 2, 3, 4, 5, 6, 7, 8, 9], 7)

In [ ]:
print("=" * 70)
print("TEST: Element not in array")
print("=" * 70)
result = binary_search_verbose([1, 2, 3, 4, 5, 6, 8, 9, 10], 7)

In [ ]:
print("=" * 60)
print("Edge Cases")
print("=" * 60)

# Empty array
print("\nEmpty array:")
print(f"binary_search_iterative([], 5) = {binary_search_iterative([], 5)}")

# Single element
print("\nSingle element (found):")
print(f"binary_search_iterative([5], 5) = {binary_search_iterative([5], 5)}")

# Single element (not found)
print("\nSingle element (not found):")
print(f"binary_search_iterative([3], 5) = {binary_search_iterative([3], 5)}")

# Compare iterative vs recursive
test_arr = [1, 3, 5, 7, 9, 11, 13, 15]
print(f"\nComparing iterative vs recursive on {test_arr}:")
for target in [1, 7, 15, 6]:
    iter_result = binary_search_iterative(test_arr, target)
    rec_result = binary_search_recursive(test_arr, target)
    print(f"  target={target:2d}: iterative={iter_result:2d}, recursive={rec_result:2d}")

---
## 3. Binary Search Variants

### 3.1 Find First Occurrence

**When to use?**
- When array has **duplicate elements**
- Need the **leftmost** position of target

**Key insight:**
- When we find target, don't stop—continue searching left
- Keep track of the last found position

#### ASCII Art Visualization

```
Find FIRST 5 in [1, 3, 5, 5, 5, 7, 9]
indices:         0  1  2  3  4  5  6

Step 1: [1, 3, 5, 5, 5, 7, 9]
         L        M        R
       
       arr[3] = 5 = target
       Found! But is it the FIRST?
       Save result=3, search LEFT: right = mid - 1 = 2

Step 2: [1, 3, 5, 5, 5, 7, 9]
         L  M  R
       
       arr[1] = 3 < 5 → search RIGHT
       left = mid + 1 = 2

Step 3: [1, 3, 5, 5, 5, 7, 9]
               LMR
       
       arr[2] = 5 = target
       Found! Update result=2, search LEFT: right = 1

Step 4: left (2) > right (1) → STOP

Answer: First occurrence at index 2
                   ↓
        [1, 3, 5, 5, 5, 7, 9]
               ↑  ↑  ↑
           first  │  last
```

In [ ]:
def find_first_occurrence(arr: List[int], target: int) -> int:
    """
    Find the first (leftmost) occurrence of target in sorted array.
    
    Args:
        arr: Sorted list of elements (may contain duplicates)
        target: Element to find
    
    Returns:
        Index of first occurrence, -1 if not found
    
    Time Complexity: O(log n)
    Space Complexity: O(1)
    """
    left, right = 0, len(arr) - 1
    result = -1
    
    while left <= right:
        mid = (left + right) // 2
        
        if arr[mid] == target:
            result = mid  # Save this position
            right = mid - 1  # Continue searching left
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return result


def find_first_occurrence_recursive(arr: List[int], target: int,
                                     left: int = None, right: int = None,
                                     result: int = -1) -> int:
    """
    Recursive version of find_first_occurrence.
    """
    if left is None:
        left = 0
    if right is None:
        right = len(arr) - 1
    
    if left > right:
        return result
    
    mid = (left + right) // 2
    
    if arr[mid] == target:
        # Found target, but continue searching left for earlier occurrence
        return find_first_occurrence_recursive(arr, target, left, mid - 1, mid)
    elif arr[mid] < target:
        return find_first_occurrence_recursive(arr, target, mid + 1, right, result)
    else:
        return find_first_occurrence_recursive(arr, target, left, mid - 1, result)


# Test
test_arr = [1, 3, 5, 5, 5, 7, 9]
print(f"Array: {test_arr}")
print(f"Find first 5: index {find_first_occurrence(test_arr, 5)}")
print(f"Find first 7: index {find_first_occurrence(test_arr, 7)}")
print(f"Find first 4: index {find_first_occurrence(test_arr, 4)}")

### 3.2 Find Last Occurrence

**Key insight:**
- When we find target, don't stop—continue searching **right**
- Mirror of find_first_occurrence

#### ASCII Art Visualization

```
Find LAST 5 in [1, 3, 5, 5, 5, 7, 9]
indices:        0  1  2  3  4  5  6

Step 1: arr[3] = 5 = target
        Save result=3, search RIGHT: left = 4

Step 2: [1, 3, 5, 5, 5, 7, 9]
                  L  M  R
        arr[5] = 7 > 5 → search LEFT

Step 3: [1, 3, 5, 5, 5, 7, 9]
                  LMR
        arr[4] = 5 = target
        Save result=4, search RIGHT: left = 5

Step 4: left (5) > right (4) → STOP

Answer: Last occurrence at index 4
                      ↓
        [1, 3, 5, 5, 5, 7, 9]
               ↑  ↑  ↑
          first   │  last
```

In [ ]:
def find_last_occurrence(arr: List[int], target: int) -> int:
    """
    Find the last (rightmost) occurrence of target in sorted array.
    
    Args:
        arr: Sorted list of elements (may contain duplicates)
        target: Element to find
    
    Returns:
        Index of last occurrence, -1 if not found
    
    Time Complexity: O(log n)
    Space Complexity: O(1)
    """
    left, right = 0, len(arr) - 1
    result = -1
    
    while left <= right:
        mid = (left + right) // 2
        
        if arr[mid] == target:
            result = mid  # Save this position
            left = mid + 1  # Continue searching right
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return result


def count_occurrences(arr: List[int], target: int) -> int:
    """
    Count occurrences of target using first and last occurrence.
    
    Time Complexity: O(log n) - two binary searches
    """
    first = find_first_occurrence(arr, target)
    if first == -1:
        return 0
    last = find_last_occurrence(arr, target)
    return last - first + 1


# Test
test_arr = [1, 3, 5, 5, 5, 7, 9]
print(f"Array: {test_arr}")
print(f"Find last 5: index {find_last_occurrence(test_arr, 5)}")
print(f"Count of 5s: {count_occurrences(test_arr, 5)}")
print(f"Count of 1s: {count_occurrences(test_arr, 1)}")
print(f"Count of 4s: {count_occurrences(test_arr, 4)}")

### 3.3 Lower Bound and Upper Bound (Insertion Point)

**Lower Bound:** First position where element could be inserted to maintain sorted order (first index ≥ target)

**Upper Bound:** Last position where element could be inserted (first index > target)

**When to use?**
- Finding insertion position for sorted container
- Range queries on sorted data
- Implementing `bisect_left` and `bisect_right`

#### ASCII Art Visualization

```
Array: [1, 3, 5, 5, 5, 7, 9]
Index:  0  1  2  3  4  5  6

lower_bound(5) = 2  (first position where 5 could go)
upper_bound(5) = 5  (position after all 5s)

        [1, 3, 5, 5, 5, 7, 9]
               ↑        ↑
           lower=2   upper=5

Insert 4:
        [1, 3, 5, 5, 5, 7, 9]
               ↑
          lower_bound(4) = 2
          upper_bound(4) = 2
          
Result: [1, 3, 4, 5, 5, 5, 7, 9]

Insert 6:
        [1, 3, 5, 5, 5, 7, 9]
                        ↑
          lower_bound(6) = 5
          upper_bound(6) = 5
          
Result: [1, 3, 5, 5, 5, 6, 7, 9]
```

In [ ]:
def lower_bound(arr: List[int], target: int) -> int:
    """
    Find the first position where target could be inserted.
    Returns index of first element >= target.
    Equivalent to C++ std::lower_bound and Python bisect.bisect_left.
    
    Args:
        arr: Sorted list of elements
        target: Element to find insertion point for
    
    Returns:
        Index where target should be inserted (0 to len(arr))
    
    Time Complexity: O(log n)
    Space Complexity: O(1)
    """
    left, right = 0, len(arr)
    
    while left < right:
        mid = (left + right) // 2
        if arr[mid] < target:
            left = mid + 1
        else:
            right = mid
    
    return left


def upper_bound(arr: List[int], target: int) -> int:
    """
    Find the position after the last occurrence of target.
    Returns index of first element > target.
    Equivalent to C++ std::upper_bound and Python bisect.bisect_right.
    
    Args:
        arr: Sorted list of elements
        target: Element to find insertion point for
    
    Returns:
        Index where target should be inserted after existing values
    
    Time Complexity: O(log n)
    Space Complexity: O(1)
    """
    left, right = 0, len(arr)
    
    while left < right:
        mid = (left + right) // 2
        if arr[mid] <= target:  # Note: <= instead of <
            left = mid + 1
        else:
            right = mid
    
    return left


# Test and compare with Python's bisect
import bisect

test_arr = [1, 3, 5, 5, 5, 7, 9]
print(f"Array: {test_arr}")
print()

for target in [0, 1, 4, 5, 6, 9, 10]:
    lb = lower_bound(test_arr, target)
    ub = upper_bound(test_arr, target)
    bisect_lb = bisect.bisect_left(test_arr, target)
    bisect_ub = bisect.bisect_right(test_arr, target)
    
    print(f"target={target}: lower_bound={lb} (bisect_left={bisect_lb}), "
          f"upper_bound={ub} (bisect_right={bisect_ub})")

### 3.4 Search in Rotated Sorted Array

**Problem:** Array was sorted but then rotated at some pivot.
Example: `[4, 5, 6, 7, 0, 1, 2]` was originally `[0, 1, 2, 4, 5, 6, 7]`

**Key insight:**
- At least one half of the array is always sorted
- Determine which half is sorted, then check if target is in that half
- If target is in sorted half → search there
- Otherwise → search the other half

#### ASCII Art Visualization

```
Find 0 in [4, 5, 6, 7, 0, 1, 2]
indices:   0  1  2  3  4  5  6

Original sorted: [0, 1, 2, 4, 5, 6, 7]
Rotated at pivot 4: [4, 5, 6, 7, 0, 1, 2]
                              ↑
                          rotation point

Step 1: [4, 5, 6, 7, 0, 1, 2]
         L        M        R
         
         arr[0]=4, arr[3]=7, arr[6]=2
         
         Is LEFT half sorted? arr[L] <= arr[M]?
         4 <= 7 → YES, left half [4,5,6,7] is sorted
         
         Is target=0 in sorted left half? 4 <= 0 <= 7?
         NO → search RIGHT half
         left = mid + 1 = 4

Step 2: [4, 5, 6, 7, 0, 1, 2]
                     L  M  R
         
         arr[4]=0, arr[5]=1, arr[6]=2
         
         Is LEFT half sorted? arr[L] <= arr[M]?
         0 <= 1 → YES, left half [0,1] is sorted
         
         Is target=0 in sorted left half? 0 <= 0 <= 1?
         YES → search LEFT half
         right = mid - 1 = 4

Step 3: [4, 5, 6, 7, 0, 1, 2]
                    LMR
         
         arr[4] = 0 = target → FOUND at index 4!
```

In [ ]:
def search_rotated_array(arr: List[int], target: int) -> int:
    """
    Search for target in a rotated sorted array (no duplicates).
    
    A rotated sorted array is an array that was originally sorted in
    ascending order, then rotated at some pivot point.
    Example: [4, 5, 6, 7, 0, 1, 2] is [0, 1, 2, 4, 5, 6, 7] rotated at index 4
    
    Args:
        arr: Rotated sorted array without duplicates
        target: Element to find
    
    Returns:
        Index of target if found, -1 otherwise
    
    Time Complexity: O(log n)
    Space Complexity: O(1)
    """
    if not arr:
        return -1
    
    left, right = 0, len(arr) - 1
    
    while left <= right:
        mid = (left + right) // 2
        
        if arr[mid] == target:
            return mid
        
        # Check if left half is sorted
        if arr[left] <= arr[mid]:
            # Left half is sorted
            if arr[left] <= target < arr[mid]:
                # Target is in the sorted left half
                right = mid - 1
            else:
                # Target is in the right half
                left = mid + 1
        else:
            # Right half is sorted
            if arr[mid] < target <= arr[right]:
                # Target is in the sorted right half
                left = mid + 1
            else:
                # Target is in the left half
                right = mid - 1
    
    return -1


def find_rotation_point(arr: List[int]) -> int:
    """
    Find the index of the minimum element (rotation point) in rotated array.
    
    Time Complexity: O(log n)
    """
    if not arr:
        return -1
    
    left, right = 0, len(arr) - 1
    
    # Array not rotated
    if arr[left] <= arr[right]:
        return 0
    
    while left <= right:
        mid = (left + right) // 2
        
        # Check if mid is the rotation point
        if mid > 0 and arr[mid] < arr[mid - 1]:
            return mid
        if mid < len(arr) - 1 and arr[mid] > arr[mid + 1]:
            return mid + 1
        
        # Decide which half to search
        if arr[left] <= arr[mid]:
            left = mid + 1
        else:
            right = mid - 1
    
    return 0

In [ ]:
def search_rotated_verbose(arr: List[int], target: int) -> int:
    """Search rotated array with detailed output."""
    print(f"Searching for {target} in rotated array {arr}")
    print("-" * 60)
    
    if not arr:
        print("Empty array!")
        return -1
    
    left, right = 0, len(arr) - 1
    step = 1
    
    while left <= right:
        mid = (left + right) // 2
        
        print(f"\nStep {step}: left={left}, mid={mid}, right={right}")
        print(f"  arr[left]={arr[left]}, arr[mid]={arr[mid]}, arr[right]={arr[right]}")
        
        if arr[mid] == target:
            print(f"  arr[{mid}] == {target} → FOUND!")
            return mid
        
        if arr[left] <= arr[mid]:
            print(f"  Left half [{arr[left]}..{arr[mid]}] is sorted")
            if arr[left] <= target < arr[mid]:
                print(f"  {arr[left]} <= {target} < {arr[mid]} → search LEFT")
                right = mid - 1
            else:
                print(f"  Target not in left half → search RIGHT")
                left = mid + 1
        else:
            print(f"  Right half [{arr[mid]}..{arr[right]}] is sorted")
            if arr[mid] < target <= arr[right]:
                print(f"  {arr[mid]} < {target} <= {arr[right]} → search RIGHT")
                left = mid + 1
            else:
                print(f"  Target not in right half → search LEFT")
                right = mid - 1
        
        step += 1
    
    print(f"\nNot found after {step-1} steps")
    return -1

# Test
rotated = [4, 5, 6, 7, 0, 1, 2]
print("=" * 70)
print("TEST: Search in rotated sorted array")
print("=" * 70)
print(f"\nRotation point: index {find_rotation_point(rotated)}\n")

result = search_rotated_verbose(rotated, 0)

In [ ]:
# More test cases for rotated array search
print("\nAll test cases:")
print("-" * 40)

test_cases = [
    ([4, 5, 6, 7, 0, 1, 2], 0),   # target in right part
    ([4, 5, 6, 7, 0, 1, 2], 6),   # target in left part
    ([4, 5, 6, 7, 0, 1, 2], 3),   # not found
    ([1], 1),                      # single element found
    ([1], 0),                      # single element not found
    ([2, 1], 1),                   # two elements
    ([1, 2, 3, 4, 5], 3),          # not rotated
    ([3, 1], 3),                   # rotation at end
]

for arr, target in test_cases:
    result = search_rotated_array(arr, target)
    print(f"search({arr}, {target}) = {result}")

---
## 4. Performance Comparison

Let's compare the performance of linear search vs binary search with real timing data.

In [ ]:
def benchmark_search(search_func, arr, targets):
    """Benchmark a search function."""
    times = []
    for target in targets:
        start = time.perf_counter()
        search_func(arr, target)
        end = time.perf_counter()
        times.append(end - start)
    return np.mean(times)


# Test arrays of increasing sizes
sizes = list(range(100, 10001, 500)) + list(range(15000, 100001, 10000))

linear_times = []
binary_times = []

print("Running benchmarks...")
for n in sizes:
    arr = list(range(n))  # Sorted array
    # Test with multiple targets (beginning, middle, end, not found)
    targets = [0, n // 4, n // 2, 3 * n // 4, n - 1, n + 1]
    
    linear_time = benchmark_search(linear_search, arr, targets)
    binary_time = benchmark_search(binary_search_iterative, arr, targets)
    
    linear_times.append(linear_time * 1000)  # Convert to ms
    binary_times.append(binary_time * 1000)
    
    if n % 20000 == 0:
        print(f"  n={n}: linear={linear_time*1000:.4f}ms, binary={binary_time*1000:.6f}ms")

print("Done!")

In [ ]:
# Plot the results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale plot
ax1 = axes[0]
ax1.plot(sizes, linear_times, 'b-o', label='Linear Search O(n)', markersize=4)
ax1.plot(sizes, binary_times, 'r-s', label='Binary Search O(log n)', markersize=4)
ax1.set_xlabel('Array Size (n)')
ax1.set_ylabel('Average Time (ms)')
ax1.set_title('Search Performance Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log scale plot to see binary search better
ax2 = axes[1]
ax2.plot(sizes, linear_times, 'b-o', label='Linear Search O(n)', markersize=4)
ax2.plot(sizes, binary_times, 'r-s', label='Binary Search O(log n)', markersize=4)
ax2.set_xlabel('Array Size (n)')
ax2.set_ylabel('Average Time (ms) - Log Scale')
ax2.set_title('Search Performance (Log Scale)')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print speedup
print("\nSpeedup of Binary Search over Linear Search:")
print("-" * 50)
for i, n in enumerate(sizes[::5]):
    idx = sizes.index(n)
    speedup = linear_times[idx] / binary_times[idx] if binary_times[idx] > 0 else float('inf')
    print(f"n={n:>6}: Binary is {speedup:>8.1f}x faster")

In [ ]:
# Theoretical vs actual comparison counts
def count_comparisons_linear(arr, target):
    """Linear search counting comparisons."""
    comparisons = 0
    for i in range(len(arr)):
        comparisons += 1
        if arr[i] == target:
            return i, comparisons
    return -1, comparisons


def count_comparisons_binary(arr, target):
    """Binary search counting comparisons."""
    comparisons = 0
    left, right = 0, len(arr) - 1
    
    while left <= right:
        mid = (left + right) // 2
        comparisons += 1
        
        if arr[mid] == target:
            return mid, comparisons
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return -1, comparisons


# Compare actual vs theoretical
print("Comparison Count Analysis")
print("=" * 60)
print(f"{'Array Size':>12} {'Linear (avg)':>14} {'Binary (max)':>14} {'log2(n)':>10}")
print("-" * 60)

for n in [10, 100, 1000, 10000, 100000]:
    arr = list(range(n))
    
    # Average linear comparisons (worst case is n)
    linear_comps = []
    binary_comps = []
    
    for target in random.sample(range(n), min(100, n)):
        _, lc = count_comparisons_linear(arr, target)
        _, bc = count_comparisons_binary(arr, target)
        linear_comps.append(lc)
        binary_comps.append(bc)
    
    print(f"{n:>12} {np.mean(linear_comps):>14.1f} {max(binary_comps):>14} {np.log2(n):>10.1f}")

---
## 5. Common Pitfalls

### 5.1 Integer Overflow in Mid Calculation

**Problem:** In languages with fixed-size integers (C, Java), `(left + right)` can overflow.

```python
# Potentially dangerous in C/Java:
mid = (left + right) / 2  # Can overflow if left + right > MAX_INT

# Safe alternative:
mid = left + (right - left) / 2  # Never overflows
```

**Note:** Python has arbitrary precision integers, so this isn't an issue in Python.

In [ ]:
# Demonstration of overflow issue (simulated)
import sys

# In Python, integers can be arbitrarily large
MAX_INT = 2**31 - 1  # Max 32-bit signed integer

left = MAX_INT - 10
right = MAX_INT

# This would overflow in C/Java
naive_mid = (left + right) // 2

# Safe calculation
safe_mid = left + (right - left) // 2

print("Integer Overflow Prevention")
print("=" * 50)
print(f"MAX_INT = {MAX_INT}")
print(f"left = {left}")
print(f"right = {right}")
print(f"left + right = {left + right}")
print(f"  → Would overflow in C/Java (max is {MAX_INT})")
print(f"\nNaive mid (left + right) // 2 = {naive_mid}")
print(f"Safe mid left + (right - left) // 2 = {safe_mid}")
print(f"\nBoth give same result in Python, but safe version works everywhere!")

### 5.2 Off-by-One Errors

The most common bugs in binary search come from boundary conditions.

In [ ]:
def binary_search_buggy_v1(arr, target):
    """BUG: Using < instead of <= causes infinite loop or misses elements."""
    left, right = 0, len(arr) - 1
    iterations = 0
    
    while left < right:  # BUG: should be <=
        iterations += 1
        if iterations > 100:
            return -2  # Infinite loop protection
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def binary_search_buggy_v2(arr, target):
    """BUG: Forgetting to update left/right properly."""
    left, right = 0, len(arr) - 1
    iterations = 0
    
    while left <= right:
        iterations += 1
        if iterations > 100:
            return -2  # Infinite loop protection
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid  # BUG: should be mid + 1
        else:
            right = mid  # BUG: should be mid - 1
    return -1


# Demonstrate the bugs
print("Off-by-One Error Demonstrations")
print("=" * 50)

arr = [1, 2, 3, 4, 5]

print("\nBug 1: Using < instead of <= in while condition")
print(f"Array: {arr}")
print(f"Correct search for 5: {binary_search_iterative(arr, 5)}")
print(f"Buggy search for 5: {binary_search_buggy_v1(arr, 5)} (misses last element!)")

print("\nBug 2: Not updating left/right properly (causes infinite loop)")
arr = [1, 2]
print(f"Array: {arr}")
result = binary_search_buggy_v2(arr, 2)
if result == -2:
    print(f"Buggy search for 2: INFINITE LOOP DETECTED!")
else:
    print(f"Buggy search for 2: {result}")

### 5.3 When to Use `<` vs `<=`

The choice depends on how you define your search space:

In [ ]:
print("Understanding < vs <= in Binary Search")
print("=" * 60)

print("""
Pattern 1: while (left <= right)
- Search space: [left, right] (inclusive both ends)
- Termination: left > right (search space is empty)
- Update: left = mid + 1 OR right = mid - 1
- Use for: Finding exact match

Pattern 2: while (left < right)
- Search space: [left, right) (left inclusive, right exclusive)
- Termination: left == right (search space has 1 element)
- Update: left = mid + 1 OR right = mid (not mid - 1!)
- Use for: Finding boundaries (lower_bound, upper_bound)
""")

# Demonstration
arr = [1, 2, 3, 4, 5]

print("Example with arr = [1, 2, 3, 4, 5], searching for 3:\n")

# Pattern 1
print("Pattern 1 (left <= right):")
left, right = 0, 4
while left <= right:
    mid = (left + right) // 2
    print(f"  left={left}, right={right}, mid={mid}, arr[mid]={arr[mid]}")
    if arr[mid] == 3:
        print(f"  Found at index {mid}!")
        break
    elif arr[mid] < 3:
        left = mid + 1
    else:
        right = mid - 1

### 5.4 Summary of Key Points

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════════════╗
║                    BINARY SEARCH CHEAT SHEET                         ║
╠══════════════════════════════════════════════════════════════════════╣
║ PREREQUISITES:                                                       ║
║   • Array must be SORTED                                             ║
║   • Need O(1) random access (arrays, not linked lists)               ║
╠══════════════════════════════════════════════════════════════════════╣
║ TEMPLATE (Standard Binary Search):                                   ║
║   left, right = 0, len(arr) - 1                                      ║
║   while left <= right:                                               ║
║       mid = left + (right - left) // 2  # Overflow-safe              ║
║       if arr[mid] == target: return mid                              ║
║       elif arr[mid] < target: left = mid + 1                         ║
║       else: right = mid - 1                                          ║
║   return -1                                                          ║
╠══════════════════════════════════════════════════════════════════════╣
║ VARIANTS:                                                            ║
║   • First occurrence: when found, save & continue LEFT               ║
║   • Last occurrence:  when found, save & continue RIGHT              ║
║   • Lower bound:      first element >= target                        ║
║   • Upper bound:      first element > target                         ║
║   • Rotated array:    check which half is sorted                     ║
╠══════════════════════════════════════════════════════════════════════╣
║ COMMON BUGS:                                                         ║
║   ✗ (left + right) overflow in C/Java                                ║
║   ✗ Using < instead of <= (misses single element)                    ║
║   ✗ left = mid or right = mid (infinite loop)                        ║
║   ✗ Not handling empty array                                         ║
╠══════════════════════════════════════════════════════════════════════╣
║ COMPLEXITY:                                                          ║
║   Time:  O(log n)     Space: O(1) iterative, O(log n) recursive      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(summary)

---
## Exercises

Try implementing these variations:

1. **Search in rotated array with duplicates** - More challenging than without duplicates
2. **Find peak element** - Element greater than its neighbors
3. **Search in 2D sorted matrix** - Each row and column is sorted
4. **Find square root** - Using binary search on answer space
5. **Minimum in rotated array** - Find the rotation point

In [ ]:
# Exercise solutions template - try implementing these yourself first!

def find_peak_element(arr: List[int]) -> int:
    """
    Find a peak element (greater than neighbors) using binary search.
    
    Time: O(log n)
    """
    if not arr:
        return -1
    if len(arr) == 1:
        return 0
    
    left, right = 0, len(arr) - 1
    
    while left < right:
        mid = (left + right) // 2
        
        if arr[mid] < arr[mid + 1]:
            # Peak is on the right side
            left = mid + 1
        else:
            # Peak is on the left side (including mid)
            right = mid
    
    return left


def sqrt_binary_search(n: int, precision: float = 1e-10) -> float:
    """
    Calculate square root using binary search.
    
    Time: O(log(n/precision))
    """
    if n < 0:
        raise ValueError("Cannot compute square root of negative number")
    if n == 0:
        return 0
    
    left, right = 0.0, max(1.0, float(n))
    
    while right - left > precision:
        mid = (left + right) / 2
        if mid * mid < n:
            left = mid
        else:
            right = mid
    
    return (left + right) / 2


# Test exercises
print("Exercise Solutions")
print("=" * 50)

print("\n1. Find Peak Element:")
test_peaks = [[1, 2, 3, 1], [1, 2, 1, 3, 5, 6, 4], [1]]
for arr in test_peaks:
    peak_idx = find_peak_element(arr)
    print(f"   {arr} → peak at index {peak_idx} (value={arr[peak_idx]})")

print("\n2. Square Root:")
for n in [2, 10, 100, 2.25]:
    result = sqrt_binary_search(n)
    import math
    print(f"   sqrt({n}) = {result:.10f} (math.sqrt = {math.sqrt(n):.10f})")

---[< Previous: Linear Time Sorting Algorithms](../02_Sorting_Algorithms/02_linear_sorts.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Linked Lists >](../04_Linear_Data_Structures/01_linked_lists.ipynb)---